# kaNAIRO — free Kaggle image-to-3D asset lab

Project rules: read `skills/kaggle-asset-lab/SKILL.md`, `skills/blender-nairobi-game-art/SKILL.md`, and `docs/BLENDER_ASSETS.md` in the cloned game repository before preparing a model for the game.

TripoSR reconstructs a 3D guess from an image; it does not make assets from text, promise correct unseen sides, create game animation, or guarantee production-ready topology. Start with one clean reference and review the result in Blender before integrating it. Only upload images you have the right to use.



In [ ]:
from pathlib import Path
import subprocess, sys, json, os

assert Path('/kaggle').exists(), 'Run this notebook on Kaggle.'
subprocess.run(['nvidia-smi'], check=False)
WORK = Path('/kaggle/working/nairoblife-assets')
WORK.mkdir(parents=True, exist_ok=True)
print('Outputs will be written to:', WORK)

## Clone the project instructions and TripoSR

The game clone is only for its project-specific skill, asset rules and provenance notes. TripoSR code and pretrained checkpoint are MIT-licensed according to its upstream README. Check the upstream license and current dependency instructions before each new run.

In [ ]:
GAME = Path('/kaggle/working/NairobLife')
TRIPOSR = Path('/kaggle/working/TripoSR')
if not GAME.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'arena/f8aec938-nairoblife', '--single-branch', 'https://github.com/wambetebenjamin/NairobLife.git', str(GAME)], check=True)
if not TRIPOSR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/VAST-AI-Research/TripoSR.git', str(TRIPOSR)], check=True)
print((GAME / 'skills/kaggle-asset-lab/SKILL.md').read_text())
print((GAME / 'skills/blender-nairobi-game-art/SKILL.md').read_text()[:5000])
print('TripoSR checkout:', subprocess.check_output(['git','-C',str(TRIPOSR),'rev-parse','HEAD'], text=True).strip())

## Install TripoSR dependencies

Kaggle already provides PyTorch in many GPU images. The upstream requirements file intentionally does not pin/install PyTorch. Dependency resolution or CUDA extension compilation can still fail when Kaggle's image changes; stop and inspect the error rather than blindly upgrading the preinstalled CUDA/PyTorch stack.

In [ ]:
%cd /kaggle/working/TripoSR
!python -m pip install -q --upgrade setuptools
!python -m pip install -q -r requirements.txt
!python -m pip install -q git+https://github.com/tatsy/torchmcubes.git
import torch
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running model inference.')

## Verify TripoSR runtime dependencies

Run this guard before the image-to-3D batch. `run.py` imports `rembg` even when model generation is called separately, so a failed/skipped requirements install must be fixed before inference.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('rembg') is None or importlib.util.find_spec('onnxruntime') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rembg[cpu]'])
import onnxruntime, rembg
print('rembg + CPU onnxruntime ready:', onnxruntime.__version__)

## Find rights-cleared Nairobi reference media

With Kaggle internet enabled, this searches **Wikimedia Commons** (not Google scraping or YouTube downloads) for Nairobi wildlife, urban scenes, and short video references. It downloads only files with identifiable CC0, public-domain, CC BY, or CC BY-SA terms, and records author/license/source metadata. CC BY requires attribution; CC BY-SA also carries ShareAlike obligations, so review its terms before using it in a derived asset. This does not auto-select a model input: inspect the downloaded references first.

In [ ]:
import hashlib, html, mimetypes, re, shutil, time
from datetime import datetime, timezone
from urllib.parse import quote
import requests

MEDIA_DIR = WORK / 'commons-media'
MEDIA_DIR.mkdir(parents=True, exist_ok=True)
COMMONS_API = 'https://commons.wikimedia.org/w/api.php'
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'kaNAIRO-KaggleAssetLab/1.0 (public educational game asset research)'})

def plain(value):
    if isinstance(value, dict): value = value.get('value', '')
    return html.unescape(re.sub(r'<[^>]+>', '', str(value or ''))).strip()

def reusable_license(label):
    text = label.lower().replace('creativecommons', 'cc')
    if 'noncommercial' in text or 'non-commercial' in text or 'no derivatives' in text or 'no-derivatives' in text or 'cc by-nc' in text or 'cc by-nd' in text:
        return False
    return ('cc0' in text or 'public domain' in text or 'cc by' in text or 'attribution' in text)

def search_commons(query, kind, max_results=2):
    filetype = 'video' if kind == 'video' else 'bitmap'
    params = {'action':'query','format':'json','generator':'search','gsrnamespace':6,'gsrlimit':30,'gsrsearch':f'{query} filetype:{filetype}','prop':'imageinfo','iiprop':'url|mime|size|extmetadata','iiurlwidth':1400}
    response = SESSION.get(COMMONS_API, params=params, timeout=60)
    response.raise_for_status()
    pages = response.json().get('query', {}).get('pages', {})
    found = []
    for page in sorted(pages.values(), key=lambda p: p.get('index', 9999)):
        info_list = page.get('imageinfo') or []
        if not info_list: continue
        info = info_list[0]
        mime = info.get('mime', '')
        if (kind == 'video' and not mime.startswith('video/')) or (kind == 'image' and not mime.startswith('image/')): continue
        ext = info.get('extmetadata', {})
        license_name = plain(ext.get('LicenseShortName')) or plain(ext.get('UsageTerms'))
        if not reusable_license(license_name): continue
        if kind == 'video' and info.get('size', 0) > 120 * 1024 * 1024: continue
        url = info.get('url') if kind == 'video' else info.get('thumburl', info.get('url'))
        found.append({'page':page,'info':info,'mime':mime,'license':license_name,'url':url,'ext':ext})
        if len(found) >= max_results: break
    return found

topics = [
    ('Nairobi National Park giraffe','image'),
    ('Nairobi National Park zebra','image'),
    ('Nairobi National Park rhinoceros','image'),
    ('Nairobi National Park wildlife','image'),
    ('Nairobi acacia Kenya','image'),
    ('Nairobi skyline Kenya','image'),
    ('Nairobi matatu street','image'),
    ('Nairobi National Park giraffe','video'),
    ('Nairobi National Park zebra','video'),
    ('Nairobi Kenya street','video'),
    ]
media_manifest = []
seen_pages = set()
for query, kind in topics:
    try:
        candidates = search_commons(query, kind, max_results=1 if kind == 'video' else 2)
    except Exception as exc:
        print('Search failed:', query, type(exc).__name__, str(exc)[:240])
        continue
    for item in candidates:
        page = item['page']; info = item['info']
        if page['pageid'] in seen_pages: continue
        seen_pages.add(page['pageid'])
        suffix = mimetypes.guess_extension(item['mime']) or ('.webm' if kind == 'video' else '.jpg')
        if suffix == '.jpe': suffix = '.jpg'
        dest = MEDIA_DIR / f"{kind}-{len(media_manifest)+1:02d}{suffix}"
        try:
            with SESSION.get(item['url'], stream=True, timeout=(30,180)) as r:
                r.raise_for_status()
                written = 0
                with dest.open('wb') as f:
                    for chunk in r.iter_content(1024*1024):
                        if not chunk: continue
                        written += len(chunk)
                        if written > (120 if kind == 'video' else 20) * 1024 * 1024: raise ValueError('download exceeds configured size cap')
                        f.write(chunk)
            record = {'kind':kind,'query':query,'title':page.get('title'),'page_url':'https://commons.wikimedia.org/wiki/'+quote(page.get('title','').replace(' ','_')),'source_url':info.get('descriptionurl'),'download_url':item['url'],'mime':item['mime'],'license':item['license'],'artist':plain(item['ext'].get('Artist')),'credit':plain(item['ext'].get('Credit')),'usage_terms':plain(item['ext'].get('UsageTerms')),'description':plain(item['ext'].get('ImageDescription'))[:1200],'file':str(dest),'bytes':dest.stat().st_size,'sha256':hashlib.sha256(dest.read_bytes()).hexdigest(),'downloaded_utc':datetime.now(timezone.utc).isoformat()}
            media_manifest.append(record)
            print(kind, record['license'], record['title'], '->', dest.name, f"({record['bytes']:,} bytes)")
        except Exception as exc:
            dest.unlink(missing_ok=True)
            print('Download skipped:', page.get('title'), type(exc).__name__, str(exc)[:200])
        time.sleep(0.15)

(MEDIA_DIR / 'media-manifest.json').write_text(json.dumps(media_manifest, indent=2, ensure_ascii=False))
print(f'Fetched {len(media_manifest)} licensed reference files into {MEDIA_DIR}')
print('Review titles and licenses; do not assume each search result depicts the query exactly.')

## Extract a few review frames from downloaded videos

Frames are visual references only. TripoSR is single-image conditioned, so choose one clean frame as its input; do not pass a video or unrelated frames as though they were multi-view reconstruction.

In [ ]:
import shutil, subprocess
FRAME_DIR = WORK / 'video-reference-frames'
FRAME_DIR.mkdir(parents=True, exist_ok=True)
frame_records = []
ffmpeg = shutil.which('ffmpeg')
if not ffmpeg:
    print('ffmpeg is not available in this Kaggle image; keep the downloaded videos as references or install a compatible ffmpeg package.')
else:
    for record in media_manifest:
        if record['kind'] != 'video': continue
        target = FRAME_DIR / (Path(record['file']).stem + '-%02d.jpg')
        cmd = [ffmpeg, '-hide_banner', '-loglevel', 'error', '-i', record['file'], '-vf', 'fps=1/8,scale=1280:-2', '-frames:v', '8', '-y', str(target)]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print('Frame extraction failed:', Path(record['file']).name, result.stderr[-400:])
            continue
        for frame in sorted(FRAME_DIR.glob(Path(record['file']).stem + '-*.jpg')):
            frame_records.append({'file':str(frame),'video_title':record['title'],'video_page_url':record['page_url'],'license':record['license'],'artist':record['artist'],'sha256':hashlib.sha256(frame.read_bytes()).hexdigest()})
(FRAME_DIR / 'frames-manifest.json').write_text(json.dumps(frame_records, indent=2, ensure_ascii=False))
print('Extracted reference frames:', len(frame_records), 'directory:', FRAME_DIR)

## Add more free 3D starter models: buildings, cars, police, people

This downloads a small curated CC0 set directly from the two already-vetted GitHub repositories. It is **not a Nairobi-specific/photorealistic pack**: the building and police-car models are low-poly prototypes. The Kenyan street/landmark references below are for custom candidate generation. Asset source, pinned commit, file URL and SHA-256 are written to a manifest.

In [ ]:
import hashlib
from pathlib import Path

PACK = WORK / 'curated-free-models'
PACK.mkdir(parents=True, exist_ok=True)
SHOOTER_COMMIT = '1009454e3be1490de7e25a14191a3f3248b06a6d'
PEOPLE_COMMIT = '7c331e647229364f3160790fe158beedf132075a'
models = [
    ('buildings','shooter','assets/buildings/k-city-commercial-building-a.glb','Generic low-poly commercial building'),
    ('buildings','shooter','assets/buildings/k-city-commercial-building-c.glb','Generic low-poly commercial building'),
    ('buildings','shooter','assets/buildings/k-city-commercial-building-skyscraper-a.glb','Generic low-poly skyscraper'),
    ('buildings','shooter','assets/buildings/k-city-industrial-building-a.glb','Generic low-poly industrial building'),
    ('vehicles','shooter','assets/vehicles/k-car-police.glb','Generic low-poly police car; not Kenyan livery'),
    ('vehicles','shooter','assets/vehicles/q-cars-cop.glb','Generic low-poly police car; not Kenyan livery'),
    ('vehicles','shooter','assets/vehicles/k-car-sedan.glb','Generic low-poly sedan'),
    ('vehicles','shooter','assets/vehicles/k-car-van.glb','Generic low-poly van; possible matatu blockout only'),
    ('people','people','models/rosa.glb','CC0 MakeHuman character; African woman'),
    ('people','people','models/brad.glb','CC0 MakeHuman character'),
    ('people','people','models/dale.glb','CC0 MakeHuman character'),
    ('people','people','models/ellie.glb','CC0 MakeHuman character'),
]
manifest = []
for category, source, upstream_path, note in models:
    if source == 'shooter':
        repo = 'Touhid-HZ/shooter-asset-library'; commit = SHOOTER_COMMIT; license_note = 'Repository asset pack documented as CC0; verify current source notes before redistribution.'
    else:
        repo = 'kimhongjae94/rubberneck-assets'; commit = PEOPLE_COMMIT; license_note = 'ASSETS.md identifies exported models as CC0 1.0; generator tooling has a separate GPL license.'
    url = f'https://raw.githubusercontent.com/{repo}/{commit}/{upstream_path}'
    dest = PACK / category / Path(upstream_path).name
    dest.parent.mkdir(parents=True, exist_ok=True)
    response = SESSION.get(url, timeout=120)
    response.raise_for_status()
    dest.write_bytes(response.content)
    manifest.append({'category':category,'file':str(dest),'bytes':dest.stat().st_size,'sha256':hashlib.sha256(response.content).hexdigest(),'note':note,'license_note':license_note,'repo':f'https://github.com/{repo}','commit':commit,'source_path':upstream_path,'source_url':url})
    print(category, dest.name, f"{dest.stat().st_size:,} bytes", '—', note)
(PACK / 'asset-pack-manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print('Curated free model pack:', PACK)

## Search additional Nairobi-specific reference media

This adds landmark exteriors, matatus, police vehicles and people to the rights-filtered Wikimedia Commons search. Results may be sparse or not match their search term exactly; check the title, image and license before using them.

In [ ]:
extra_topics = [
    ('Kenyatta International Convention Centre Nairobi exterior','image'),
    ('Times Tower Nairobi exterior','image'),
    ('Nairobi matatu Kenya','image'),
    ('Kenya police vehicle Nairobi','image'),
    ('Kenya police car','image'),
    ('Kenyan people Nairobi street','image'),
    ('Nairobi National Park giraffe','image'),
    ('Nairobi National Park zebra','image'),
    ('Nairobi National Park black rhinoceros','image'),
    ('Nairobi National Park African buffalo','image'),
    ('Nairobi National Park wildebeest','image'),
    ('Nairobi National Park ostrich','image'),
    ('Nairobi National Park lion','image'),
    ('Nairobi National Park cheetah','image'),
    ('Nairobi National Park eland','image'),
    ('Nairobi National Park warthog','image'),
    ('Nairobi National Park impala','image'),
    ('Nairobi National Park baboon','image'),
    ('umbrella thorn acacia Kenya','image'),
    ('jacaranda tree Nairobi Kenya','image'),
    ('Nairobi street trees Kenya','image'),
    ('Kenya fever tree savannah','image'),
    ('JKIA Nairobi airport terminal exterior','image'),
    ('Nairobi Expressway Nairobi elevated road','image'),
    ('Nyayo National Stadium Nairobi exterior','image'),
    ('Kasarani Stadium Nairobi exterior','image'),
    ('Talanta Sports City Stadium Nairobi','image'),
    ('Nairobi SGR station Madaraka Express train','image'),
    ('Nairobi National Park wildlife','video'),
    ('Nairobi giraffe Kenya','video'),
    ('Nairobi matatu Kenya','video'),
    ('Nairobi SGR Madaraka Express train Kenya','video'),
]
known_titles = {r.get('title') for r in media_manifest}
for query, kind in extra_topics:
    try: candidates = search_commons(query, kind, max_results=1)
    except Exception as exc:
        print('Search failed:', query, str(exc)[:180]); continue
    for item in candidates:
        page, info = item['page'], item['info']
        if page.get('title') in known_titles: continue
        suffix = mimetypes.guess_extension(item['mime']) or ('.webm' if kind == 'video' else '.jpg')
        if suffix == '.jpe': suffix = '.jpg'
        dest = MEDIA_DIR / f"{kind}-extra-{len(media_manifest)+1:02d}{suffix}"
        try:
            with SESSION.get(item['url'], stream=True, timeout=(30,180)) as r:
                r.raise_for_status(); written = 0
                with dest.open('wb') as f:
                    for chunk in r.iter_content(1024*1024):
                        if not chunk: continue
                        written += len(chunk)
                        if written > (120 if kind == 'video' else 20)*1024*1024: raise ValueError('file exceeds size cap')
                        f.write(chunk)
            rec = {'kind':kind,'query':query,'title':page.get('title'),'page_url':'https://commons.wikimedia.org/wiki/'+quote(page.get('title','').replace(' ','_')),'source_url':info.get('descriptionurl'),'download_url':item['url'],'mime':item['mime'],'license':item['license'],'artist':plain(item['ext'].get('Artist')),'credit':plain(item['ext'].get('Credit')),'usage_terms':plain(item['ext'].get('UsageTerms')),'description':plain(item['ext'].get('ImageDescription'))[:1200],'file':str(dest),'bytes':dest.stat().st_size,'sha256':hashlib.sha256(dest.read_bytes()).hexdigest(),'downloaded_utc':datetime.now(timezone.utc).isoformat()}
            media_manifest.append(rec); known_titles.add(page.get('title'))
            print('Added', kind, item['license'], page.get('title'), '->', dest.name)
        except Exception as exc:
            dest.unlink(missing_ok=True); print('Skipped', page.get('title'), str(exc)[:180])
        time.sleep(0.15)
(MEDIA_DIR / 'media-manifest.json').write_text(json.dumps(media_manifest, indent=2, ensure_ascii=False))
print('Reference files now recorded:', len(media_manifest))


## Generate several independent candidate models in one TripoSR run

TripoSR only reconstructs one object from each image; this batch call feeds several separately reviewed licensed images to the model in one process. It does not merge them into one scene or guarantee correct geometry. The model outputs go into numbered folders and are mapped back to their reference title/license below. Edit out any image whose result is ambiguous before running.

In [ ]:
batch_queries = {
    'wildlife-giraffe': ('wildlife','Nairobi National Park giraffe'),
    'wildlife-zebra': ('wildlife','Nairobi National Park zebra'),
    'wildlife-black-rhino': ('wildlife','Nairobi National Park black rhinoceros'),
    'wildlife-buffalo': ('wildlife','Nairobi National Park African buffalo'),
    'wildlife-wildebeest': ('wildlife','Nairobi National Park wildebeest'),
    'wildlife-ostrich': ('wildlife','Nairobi National Park ostrich'),
    'wildlife-lion': ('wildlife','Nairobi National Park lion'),
    'wildlife-cheetah': ('wildlife','Nairobi National Park cheetah'),
    'wildlife-eland': ('wildlife','Nairobi National Park eland'),
    'wildlife-warthog': ('wildlife','Nairobi National Park warthog'),
    'wildlife-impala': ('wildlife','Nairobi National Park impala'),
    'wildlife-baboon': ('wildlife','Nairobi National Park baboon'),
    'tree-acacia': ('trees','umbrella thorn acacia Kenya'),
    'tree-jacaranda': ('trees','jacaranda tree Nairobi Kenya'),
    'tree-street': ('trees','Nairobi street trees Kenya'),
    'tree-fever': ('trees','Kenya fever tree savannah'),
    'landmark-kicc': ('transport','Kenyatta International Convention Centre Nairobi exterior'),
    'landmark-times-tower': ('transport','Times Tower Nairobi exterior'),
    'transport-matatu': ('transport','Nairobi matatu Kenya'),
    'transport-police': ('transport','Kenya police vehicle Nairobi'),
    'transport-jkia': ('transport','JKIA Nairobi airport terminal exterior'),
    'transport-expressway': ('transport','Nairobi Expressway Nairobi elevated road'),
    'stadium-nyayo': ('transport','Nyayo National Stadium Nairobi exterior'),
    'stadium-kasarani': ('transport','Kasarani Stadium Nairobi exterior'),
    'stadium-talanta': ('transport','Talanta Sports City Stadium Nairobi'),
    'transport-sgr': ('transport','Nairobi SGR station Madaraka Express train'),
}
# Set to ['wildlife','trees','transport'] for everything, or one group to conserve Kaggle time.
GROUPS_TO_GENERATE = ['wildlife','trees','transport']
record_by_query = {r.get('query'):r for r in media_manifest if r.get('kind') == 'image'}
batch = [(name, group, record_by_query[query]) for name,(group,query) in batch_queries.items() if group in GROUPS_TO_GENERATE and query in record_by_query]
for name,group,r in batch: print(f"{group}: {name} => {r['title']} | {r['license']} | {r['file']}")
if not batch:
    print('No matching licensed references found. Run the Commons search cells and review their result titles first.')
else:
    BATCH_OUT = WORK / 'batch-candidates'
    BATCH_OUT.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, 'run.py'] + [r['file'] for _,_,r in batch] + ['--device','cuda:0','--model-save-format','glb','--bake-texture','--texture-resolution','1024','--output-dir',str(BATCH_OUT)]
    print('One TripoSR process will reconstruct', len(batch), 'separate reference images.')
    subprocess.run(cmd, cwd=TRIPOSR, check=True)
    batch_manifest=[]
    for i,(name,group,r) in enumerate(batch):
        mesh = BATCH_OUT / str(i) / 'mesh.glb'
        if mesh.exists():
            named = BATCH_OUT / 'named-models' / f'{name}.glb'; named.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(mesh,named)
            batch_manifest.append({'candidate':name,'group':group,'generated_glb':str(named),'reference_title':r['title'],'reference_page':r['page_url'],'license':r['license'],'artist':r['artist'],'source_image':r['file']})
    (BATCH_OUT / 'batch-manifest.json').write_text(json.dumps(batch_manifest, indent=2, ensure_ascii=False))
    print('Generated candidate GLBs:', len(batch_manifest), 'under', BATCH_OUT / 'named-models')


## Select one reference image

Choose a reviewed image from `/kaggle/input`, the downloaded Commons image folder, or the extracted video frames listed by the cell below, then paste that exact path into `IMAGE`. Prefer a single, isolated object on a plain background; whole scenes and multiple objects produce poor game meshes. For a car, use a clear three-quarter view; for a standing person, use a full-body view with limbs visible. The model still infers unseen sides from that one view.

In [ ]:
from pathlib import Path
from PIL import Image

inputs = list(Path('/kaggle/input').rglob('*')) if Path('/kaggle/input').exists() else []
web_refs = list(MEDIA_DIR.rglob('*')) + list(FRAME_DIR.rglob('*'))
images = [p for p in inputs + web_refs if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.webp'}]
print('Kaggle inputs, Commons images, and extracted video reference frames:')
for p in images[:40]: print(p)
# CHANGE this to one image listed above, from a source you are allowed to use.
IMAGE = Path('/kaggle/input/your-reference-dataset/your-reference.png')
if not IMAGE.is_file():
    raise FileNotFoundError('Set IMAGE to a real image path under /kaggle/input, then rerun this cell.')
display(Image.open(IMAGE).convert('RGB').resize((512,512)))

## Generate one GLB candidate

This asks TripoSR for a 1K baked texture and binary glTF output. If texture baking runs out of memory, retry once without `--bake-texture`; the default output uses vertex colors. Do not bulk-generate until the first model has been inspected.

In [ ]:
import datetime, shlex
ASSET_NAME = 'candidate'  # e.g. 'nairobi-hatchback' or 'pedestrian-rosa'
OUT = WORK / ASSET_NAME
OUT.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'run.py', str(IMAGE), '--device', 'cuda:0', '--model-save-format', 'glb', '--bake-texture', '--texture-resolution', '1024', '--output-dir', str(OUT)]
print('Running:', shlex.join(cmd))
subprocess.run(cmd, cwd=TRIPOSR, check=True)
glbs = sorted(OUT.rglob('*.glb'))
print('GLB outputs:', glbs)
assert glbs, 'No GLB output found; inspect the inference log and TripoSR docs.'
manifest = {'asset_name': ASSET_NAME, 'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'source_image_name': IMAGE.name, 'source_repo': 'https://github.com/VAST-AI-Research/TripoSR', 'source_commit': subprocess.check_output(['git','-C',str(TRIPOSR),'rev-parse','HEAD'], text=True).strip(), 'model': 'stabilityai/TripoSR', 'license_note': 'TripoSR README states code and pretrained model are MIT; verify current terms before distribution.', 'review_status': 'candidate only; inspect orientation, scale, topology, materials, rights and game performance before use'}
(OUT / 'asset-manifest.json').write_text(json.dumps(manifest, indent=2))
for p in glbs: print(p, p.stat().st_size, 'bytes')

## Review and bring back to the game

Kaggle output is not game-ready simply because it is GLB. Inspect in Blender: check dimensions in metres, ground contact/origin, Y-up and forward -Z, normals, topology, texture, transparency and visual quality. The game skill targets under 2 MB per hero GLB where practical and requires a Three.js fallback. TripoSR does not rig people or create animation clips, so do not use its output to replace the animated player/crowd without a separate rigging/animation pass.

Download only the reviewed `.glb` and `asset-manifest.json` from Kaggle outputs. Add it to `public/assets/` only after it has been integrated and tested in the actual game; update `docs/FREE_ASSETS.md` with provenance. Never copy Kaggle checkpoints, caches, arbitrary input photos, or tokens into Git.

## Create one ZIP from all scattered asset outputs

This cell searches recursively through `/kaggle/working` for generated 3D models, reference images/videos, extracted frames, and provenance manifests, while skipping source-code repositories and caches. It places them into one ZIP and makes one relative download link. You do not need to select or zip files one by one.


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import os, zipfile
ROOT = Path('/kaggle/working')
bundle = ROOT / 'all-nairoblife-assets.zip'
asset_exts = {'.glb','.gltf','.obj','.mtl','.png','.jpg','.jpeg','.webp','.gif','.mp4','.webm','.mov','.mkv'}
skip_dirs = {'.git','.cache','node_modules','.venv','venv','__pycache__','checkpoints'}
files = []
for p in ROOT.rglob('*'):
    if not p.is_file() or p == bundle or p.suffix.lower() == '.zip': continue
    rel = p.relative_to(ROOT)
    if any(part in skip_dirs for part in rel.parts): continue
    # Exclude cloned source trees, but keep their generated output folders if used.
    if any(part in {'NairobLife','TripoSR'} for part in rel.parts) and 'output' not in rel.parts: continue
    if p.suffix.lower() in asset_exts or ('manifest' in p.name.lower() and p.suffix.lower() == '.json'):
        files.append(p)
files = sorted(set(files))
if not files:
    print('No generated assets found under /kaggle/working. Check where the notebook saved its outputs.')
else:
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for p in files: z.write(p, p.relative_to(ROOT))
    print(f'Packed {len(files)} files into {bundle} ({bundle.stat().st_size:,} bytes).')
    os.chdir(ROOT)
    display(FileLink('all-nairoblife-assets.zip'))


## Make a small model-only ZIP for review/upload

The reference-media archive can be large. For sharing the actual game candidates, use this smaller bundle: GLBs plus manifests only (no downloaded videos or source-photo collection).

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import os, zipfile
ROOT = Path('/kaggle/working')
WORK = ROOT / 'nairoblife-assets'
bundle = ROOT / 'nairoblife-models-only.zip'
files = sorted(p for p in WORK.rglob('*') if p.is_file() and (p.suffix.lower() in {'.glb','.gltf','.obj','.mtl'} or ('manifest' in p.name.lower() and p.suffix.lower()=='.json')))
if not files:
    print('No models/manifests found. Run the model download or generation cells first.')
else:
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as z:
        for p in files: z.write(p, p.relative_to(WORK))
    print(f'Packed {len(files)} model/provenance files: {bundle.stat().st_size:,} bytes')
    if bundle.stat().st_size > 23*1024*1024:
        print('This exceeds the 25 MB chat attachment limit; reduce candidate GLBs or split by model category before attaching.')
    os.chdir(ROOT)
    display(FileLink('nairoblife-models-only.zip'))

## One-cell: fetch and ZIP all model files automatically

Run this single cell to fetch the curated free GLBs directly from the pinned public repositories, include any GLBs already generated in this Kaggle workspace plus the two GLBs in the game clone, write one provenance manifest, and create one model-only ZIP. It does not require the earlier manual download or generation cells. It intentionally omits reference photos/videos from the ZIP; those are not game-ready models and made the previous archive large.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
from IPython.display import FileLink, display
import hashlib, json, os, zipfile

ROOT = Path('/kaggle/working')
WORK = ROOT / 'nairoblife-assets'
WORK.mkdir(parents=True, exist_ok=True)
SHOOTER_COMMIT = '1009454e3be1490de7e25a14191a3f3248b06a6d'
PEOPLE_COMMIT = '7c331e647229364f3160790fe158beedf132075a'
catalog = [
    ('buildings','shooter','assets/buildings/k-city-commercial-building-a.glb','generic low-poly commercial building','CC0 per source repository asset catalog'),
    ('buildings','shooter','assets/buildings/k-city-commercial-building-c.glb','generic low-poly commercial building','CC0 per source repository asset catalog'),
    ('buildings','shooter','assets/buildings/k-city-commercial-building-skyscraper-a.glb','generic low-poly skyscraper','CC0 per source repository asset catalog'),
    ('buildings','shooter','assets/buildings/k-city-industrial-building-a.glb','generic low-poly industrial building','CC0 per source repository asset catalog'),
    ('vehicles','shooter','assets/vehicles/k-car-police.glb','generic low-poly police car, not Kenyan livery','CC0 per source repository asset catalog'),
    ('vehicles','shooter','assets/vehicles/q-cars-cop.glb','generic low-poly police car, not Kenyan livery','CC0 per source repository asset catalog'),
    ('vehicles','shooter','assets/vehicles/k-car-sedan.glb','generic low-poly sedan','CC0 per source repository asset catalog'),
    ('vehicles','shooter','assets/vehicles/k-car-van.glb','generic low-poly van, blockout only','CC0 per source repository asset catalog'),
    ('people','people','models/rosa.glb','MakeHuman character: African woman','CC0 exported model per ASSETS.md'),
    ('people','people','models/brad.glb','MakeHuman character','CC0 exported model per ASSETS.md'),
    ('people','people','models/dale.glb','MakeHuman character','CC0 exported model per ASSETS.md'),
    ('people','people','models/ellie.glb','MakeHuman character','CC0 exported model per ASSETS.md'),
]
manifest = []
for category, source, source_path, note, license_note in catalog:
    if source == 'shooter': repo='Touhid-HZ/shooter-asset-library'; commit=SHOOTER_COMMIT
    else: repo='kimhongjae94/rubberneck-assets'; commit=PEOPLE_COMMIT
    url=f'https://raw.githubusercontent.com/{repo}/{commit}/{source_path}'
    dest=WORK/'curated-free-models'/category/Path(source_path).name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists() or dest.stat().st_size == 0:
        req=Request(url, headers={'User-Agent':'kaNAIRO-KaggleAssetPack/1.0'})
        with urlopen(req, timeout=120) as response: data=response.read()
        if len(data) > 20*1024*1024: raise ValueError(f'Unexpectedly large model: {source_path}')
        dest.write_bytes(data)
    digest=hashlib.sha256(dest.read_bytes()).hexdigest()
    manifest.append({'category':category,'file':str(dest),'bytes':dest.stat().st_size,'sha256':digest,'note':note,'license_note':license_note,'source_repo':f'https://github.com/{repo}','source_commit':commit,'source_path':source_path,'source_url':url})
    print('Ready:',category,dest.name,f'{dest.stat().st_size:,} bytes')

manifest_path=WORK/'automatic-model-pack-manifest.json'
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
search_roots=[WORK, ROOT/'NairobLife'/'public'/'assets', ROOT/'TripoSR'/'output']
model_files=set()
for folder in search_roots:
    if folder.exists(): model_files.update(p for p in folder.rglob('*.glb') if p.is_file())
    if folder.exists(): model_files.update(p for p in folder.rglob('*.gltf') if p.is_file())
bundle=ROOT/'nairoblife-models.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=9) as archive:
    for path in sorted(model_files):
        try: archive.write(path,path.relative_to(ROOT))
        except ValueError: archive.write(path,Path('extra-assets')/path.name)
    for source_manifest in sorted(WORK.rglob('*manifest.json')):
        if source_manifest.is_file() and source_manifest != manifest_path:
            archive.write(source_manifest,source_manifest.relative_to(ROOT))
    archive.write(manifest_path,manifest_path.relative_to(ROOT))
print(f'ZIP ready: {len(model_files)} model files plus license/source manifest; {bundle.stat().st_size:,} bytes')
if bundle.stat().st_size > 23*1024*1024: print('Warning: ZIP is over the 25 MB chat attachment limit; use the existing model-only pack or split by category.')
os.chdir(ROOT)
display(FileLink('nairoblife-models.zip'))